In [1]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

# ── Plot styling ──────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13, "axes.labelsize": 11})
PALETTE = {"Low Activity": "#4C9BE8", "Moderate Activity": "#F5A623", "High Activity": "#E85C5C"}

# ── Load data ─────────────────────────────────────────────────────────────────
DB_PATH = "ProjectData/gas_monitoring.db"          # relative path used by ML pipeline

conn = sqlite3.connect(DB_PATH)
df_raw = pd.read_sql("SELECT * FROM gas_monitoring", conn)
conn.close()

print(f"Dataset loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
df_raw.head()


Matplotlib is building the font cache; this may take a moment.


ModuleNotFoundError: No module named 'seaborn'

In [ ]:
# ── Schema & dtypes ───────────────────────────────────────────────────────────
print("=== Column Types ===")
print(df_raw.dtypes)
print(f"\nNumerical columns : {df_raw.select_dtypes(include='number').columns.tolist()}")
print(f"Categorical columns: {df_raw.select_dtypes(include='object').columns.tolist()}")


In [ ]:
# ── Descriptive statistics (numerical) ───────────────────────────────────────
df_raw.describe().round(2)


### Conclusion — Dataset Overview
- **10,000 rows** with **14 features**: 9 numerical sensor readings, 1 integer (Session ID), 4 categorical.  
- The target column is **`Activity Level`** — a multiclass label.  
- `describe()` already hints at problems: Temperature max = 307°C and Humidity min = −49% are physically impossible, flagging contaminated/synthetic data as warned in the problem statement.  
- Session ID is an identifier and will be **dropped** from modelling features.


---
<a id='3'></a>
## 3. Data Quality Assessment

### Purpose
Systematically identify and quantify every data quality issue so that each can be addressed with a justified cleaning strategy in the pipeline. Issues include: missing values, inconsistent/dirty labels, outliers, and impossible sensor readings.


### 3.1 Missing Values

In [ ]:
# ── Missing value counts & percentages ───────────────────────────────────────
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
missing_df = missing_df[missing_df["Missing Count"] > 0].sort_values("Missing %", ascending=False)
print(missing_df)


In [ ]:
# ── Missing value heatmap ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 3))
sns.heatmap(df_raw.isnull().T, cbar=False, yticklabels=True, xticklabels=False,
            cmap="Reds", ax=ax)
ax.set_title("Missing Value Map (red = missing)")
ax.set_xlabel("Rows (10,000 observations)")
plt.tight_layout()
plt.show()


**Conclusion — Missing Values**  
Four columns have missing data:
- `Humidity` — **19.3%** missing (highest; likely sensor dropout)  
- `MetalOxideSensor_Unit2` — **14.1%** missing  
- `Ambient Light Level` — **10.5%** missing  
- `CO_GasSensor` — **8.3%** missing  

**Imputation strategy (to be applied in pipeline):**  
- Numerical columns → **median imputation** (robust to outliers present in this dataset)  
- `Ambient Light Level` → **mode imputation** (categorical)  
- These choices will be clearly documented in the pipeline config.


### 3.2 Inconsistent/Dirty Labels

In [ ]:
# ── Activity Level raw counts ─────────────────────────────────────────────────
print("=== Activity Level (raw) ===")
print(df_raw["Activity Level"].value_counts())

print("\n=== HVAC Operation Mode (raw) ===")
print(df_raw["HVAC Operation Mode"].value_counts())


In [ ]:
# ── Normalise Activity Level ──────────────────────────────────────────────────
activity_map = {
    "Low Activity":      "Low Activity",
    "Low_Activity":      "Low Activity",
    "LowActivity":       "Low Activity",
    "Moderate Activity": "Moderate Activity",
    "ModerateActivity":  "Moderate Activity",
    "High Activity":     "High Activity",
}
df = df_raw.copy()
df["Activity Level"] = df["Activity Level"].map(activity_map)

# ── Normalise HVAC Operation Mode (lowercase) ─────────────────────────────────
df["HVAC Operation Mode"] = df["HVAC Operation Mode"].str.strip().str.lower().str.replace("_", " ")

print("Activity Level after normalisation:")
print(df["Activity Level"].value_counts())
print("\nHVAC modes after normalisation:")
print(df["HVAC Operation Mode"].value_counts())


**Conclusion — Dirty Labels**  
- `Activity Level` had **6 variants** representing only 3 true classes. Inconsistent formatting (`Low_Activity`, `LowActivity`) likely arises from multiple data collection systems.  
- `HVAC Operation Mode` similarly had **23 raw variants** due to mixed casing and underscores, reduced to **6 canonical modes** after normalisation.  
- **Assumption:** Variants with identical semantics are merged. No data is discarded.


### 3.3 Outliers & Impossible Values

In [ ]:
# ── Temperature outlier analysis ──────────────────────────────────────────────
temp_outliers = df[df["Temperature"] > 100]
hum_outliers  = df[(df["Humidity"] < 0) | (df["Humidity"] > 100)]

print(f"Temperature readings > 100°C : {len(temp_outliers):,}  ({len(temp_outliers)/len(df)*100:.1f}%)")
print(f"Impossible Humidity readings : {len(hum_outliers):,}  ({len(hum_outliers)/len(df)*100:.1f}%)")

# IQR-based outlier detection for all numerical sensors
numerical_cols = ["Temperature", "Humidity", "CO2_InfraredSensor", "CO2_ElectroChemicalSensor",
                  "MetalOxideSensor_Unit1", "MetalOxideSensor_Unit2", "MetalOxideSensor_Unit3",
                  "MetalOxideSensor_Unit4", "CO_GasSensor"]

print("\n=== IQR Outlier Summary ===")
for col in numerical_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_out = ((df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)).sum()
    print(f"  {col:<35} {n_out:>5} outliers ({n_out/df[col].notna().sum()*100:.1f}%)")


In [ ]:
# ── Outlier visualisation — Temperature & Humidity ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df["Temperature"], bins=80, color="#4C9BE8", edgecolor="white", linewidth=0.4)
axes[0].axvline(100, color="red", linestyle="--", linewidth=1.5, label="Physical max (100°C)")
axes[0].set_title("Temperature Distribution (with outliers)")
axes[0].set_xlabel("Temperature (°C)")
axes[0].set_ylabel("Count")
axes[0].legend()

axes[1].hist(df["Humidity"].dropna(), bins=80, color="#F5A623", edgecolor="white", linewidth=0.4)
axes[1].axvline(0,   color="red",  linestyle="--", linewidth=1.5, label="Physical min (0%)")
axes[1].axvline(100, color="red",  linestyle="--", linewidth=1.5, label="Physical max (100%)")
axes[1].set_title("Humidity Distribution (with outliers)")
axes[1].set_xlabel("Relative Humidity (%)")
axes[1].legend()

plt.suptitle("Physically Impossible Sensor Readings", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


**Conclusion — Outliers**  
- **795 temperature readings** exceed 100°C — physically impossible for an indoor environment. These are contaminated/synthetic data points.  
- **207 humidity readings** are negative and **207 exceed 100%** — both physically impossible.  
- IQR analysis confirms additional statistical outliers across sensor columns.  

**Treatment strategy:**  
- Temperature: cap at 40°C upper bound (reasonable indoor max); values > 40°C treated as sensor error → replaced with median  
- Humidity: cap within [0, 100]; values outside this range replaced with median  
- Other sensors: IQR-based capping (Winsorisation at 1.5×IQR) to preserve data volume  
- **Assumption:** These are measurement/recording errors, not genuine readings. Removing rows would discard 8% of data unnecessarily; capping is preferred.


### 3.4 Class Distribution & Imbalance

In [ ]:
# ── Class distribution bar chart ──────────────────────────────────────────────
class_counts = df["Activity Level"].value_counts().reindex(["Low Activity", "Moderate Activity", "High Activity"])
class_pct    = class_counts / class_counts.sum() * 100

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(class_counts.index, class_counts.values,
              color=[PALETTE[c] for c in class_counts.index], edgecolor="white", linewidth=0.5)
for bar, pct in zip(bars, class_pct):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
            f"{pct:.1f}%", ha="center", fontsize=11, fontweight="bold")
ax.set_title("Activity Level Class Distribution")
ax.set_ylabel("Count")
ax.set_xlabel("Activity Level")
plt.tight_layout()
plt.show()

print(class_counts)


**Conclusion — Class Imbalance**  
| Class | Count | % |
|---|---|---|
| Low Activity | 5,767 | 57.7% |
| Moderate Activity | 3,138 | 31.4% |
| High Activity | 1,095 | 10.9% |

Significant imbalance exists, particularly for **High Activity** (10.9%). A model trained naively will bias predictions toward Low Activity.  
**Mitigation strategy (pipeline):** Use `class_weight='balanced'` in sklearn models and **SMOTE** oversampling during training. Evaluation will use **weighted F1-score** rather than accuracy.


---
<a id='4'></a>
## 4. Univariate Analysis

### Purpose
Examine each feature independently to understand its distribution, range, and central tendency. This informs decisions about scaling, transformation, and encoding in the pipeline.


### 4.1 Numerical Sensor Distributions

In [ ]:
# ── Apply domain-level cleaning for visualisation ─────────────────────────────
df_clean = df.copy()
df_clean.loc[df_clean["Temperature"] > 40, "Temperature"] = np.nan
df_clean.loc[(df_clean["Humidity"] < 0) | (df_clean["Humidity"] > 100), "Humidity"] = np.nan
df_clean["Temperature"].fillna(df_clean["Temperature"].median(), inplace=True)
df_clean["Humidity"].fillna(df_clean["Humidity"].median(), inplace=True)
for col in ["MetalOxideSensor_Unit2", "CO_GasSensor"]:
    df_clean[col].fillna(df_clean[col].median(), inplace=True)
df_clean["Ambient Light Level"].fillna(df_clean["Ambient Light Level"].mode()[0], inplace=True)

sensor_cols = ["Temperature", "Humidity", "CO2_InfraredSensor", "CO2_ElectroChemicalSensor",
               "MetalOxideSensor_Unit1", "MetalOxideSensor_Unit2",
               "MetalOxideSensor_Unit3", "MetalOxideSensor_Unit4", "CO_GasSensor"]

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes = axes.flatten()
colors = sns.color_palette("muted", len(sensor_cols))

for i, col in enumerate(sensor_cols):
    ax = axes[i]
    data = df_clean[col].dropna()
    ax.hist(data, bins=50, color=colors[i], edgecolor="white", linewidth=0.4, alpha=0.85)
    ax.axvline(data.mean(),   color="crimson",   linestyle="--", linewidth=1.2, label=f"Mean={data.mean():.1f}")
    ax.axvline(data.median(), color="steelblue", linestyle=":",  linewidth=1.2, label=f"Median={data.median():.1f}")
    ax.set_title(col, fontsize=10)
    ax.set_xlabel("Value")
    ax.set_ylabel("Count")
    ax.legend(fontsize=7)

plt.suptitle("Univariate Distributions of Sensor Readings (post domain-clipping)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


**Conclusion — Sensor Distributions**  
- **Temperature & Humidity** are approximately normally distributed after removing impossible values, centred around ~20°C and ~52% RH — consistent with indoor living spaces.  
- **CO2 sensors** show right-skewed distributions, reflecting occasional spikes from resident activity (cooking, breathing in confined spaces).  
- **Metal Oxide Sensors** (Units 1–4) vary in range; Units 2 and 4 show wider spread, suggesting they capture more variance in VOC activity — aligning with their higher correlation to Activity Level found later.  
- **CO_GasSensor** is discrete-like (values 0–4), suggesting it may have been binned or quantised.  
- Features will be **standardised (StandardScaler)** before training models sensitive to scale (Logistic Regression, SVM).


### 4.2 Categorical Feature Distributions

In [ ]:
cat_cols = ["Time of Day", "HVAC Operation Mode", "Ambient Light Level"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col in zip(axes, cat_cols):
    counts = df_clean[col].value_counts()
    counts.plot(kind="bar", ax=ax, color=sns.color_palette("muted", len(counts)), edgecolor="white")
    ax.set_title(f"{col}")
    ax.set_xlabel("")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=30)
    for p in ax.patches:
        ax.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width()/2, p.get_height()),
                    ha="center", va="bottom", fontsize=8)

plt.suptitle("Categorical Feature Distributions", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


**Conclusion — Categorical Features**  
- **Time of Day** is roughly balanced across the 4 periods (morning/afternoon/evening/night ~25% each) — good for modelling as no single period dominates.  
- **HVAC Operation Mode** is also relatively balanced across 6 modes after label normalisation — no encoding issue.  
- **Ambient Light Level** is skewed toward `very_bright` and `bright` — likely reflecting daytime activity patterns in the monitored homes.  
- All categorical features will be **one-hot encoded** in the pipeline.


---
<a id='5'></a>
## 5. Bivariate & Multivariate Analysis

### Purpose
Examine how individual features relate to the target variable (`Activity Level`) and to each other. This reveals which sensors are most informative for prediction and whether multicollinearity exists.


### 5.1 Sensor Readings by Activity Level (Violin Plots)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(sensor_cols):
    ax = axes[i]
    sns.violinplot(data=df_clean, x="Activity Level",
                   y=col, order=["Low Activity", "Moderate Activity", "High Activity"],
                   palette=PALETTE, inner="quartile", ax=ax)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=15)

plt.suptitle("Sensor Value Distributions by Activity Level", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


**Conclusion — Sensor vs Activity Level**  
- **MetalOxideSensor_Unit2 & Unit4** show the clearest separation across activity levels — High Activity residents produce higher VOC/gas readings, likely from increased movement and cooking.  
- **CO2_ElectroChemicalSensor** increases with activity level, reflecting higher CO2 from respiratory output during physical activity.  
- **CO_GasSensor** shows an *inverse* pattern — higher readings correlate with lower activity, which may indicate that low-activity periods coincide with cooking (CO release) while residents are sedentary.  
- **Temperature and Humidity** show minimal separation — they are poor standalone predictors of activity level.


### 5.2 Activity Level by Categorical Features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, ["Time of Day", "HVAC Operation Mode", "Ambient Light Level"]):
    ct = pd.crosstab(df_clean[col], df_clean["Activity Level"],
                     normalize="index")[["Low Activity", "Moderate Activity", "High Activity"]] * 100
    ct.plot(kind="bar", stacked=True, ax=ax,
            color=[PALETTE[c] for c in ct.columns], edgecolor="white", linewidth=0.3)
    ax.set_title(f"Activity Level % by {col}")
    ax.set_ylabel("Percentage (%)")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
    ax.legend(fontsize=8, loc="upper right")

plt.suptitle("Activity Level Composition by Categorical Features", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


**Conclusion — Categorical vs Activity Level**  
- **Time of Day:** High Activity is notably more prevalent in the afternoon and evening — consistent with elderly residents being more active outside of sleeping hours. Night shows the highest proportion of Low Activity.  
- **HVAC Mode:** `ventilation_only` mode coincides with slightly more High Activity (possible causation: residents open windows / increase air circulation when active).  
- **Ambient Light Level:** `very_bright` and `bright` conditions correlate with higher activity — residents are more active when the environment is well-lit, consistent with daytime hours.  
- These categorical features carry **useful signal** and should be retained and one-hot encoded.


### 5.3 Correlation Heatmap

In [ ]:
# ── Correlation heatmap (numerical features only) ─────────────────────────────
corr_df = df_clean[sensor_cols].corr()

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr_df, dtype=bool))
sns.heatmap(corr_df, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            annot_kws={"size": 9})
ax.set_title("Sensor Feature Correlation Heatmap", fontsize=13)
plt.tight_layout()
plt.show()


**Conclusion — Feature Correlations**  
- **CO2_InfraredSensor and CO2_ElectroChemicalSensor** are highly correlated (r ≈ 0.85+), suggesting they measure the same underlying phenomenon. One may be dropped or PCA applied to reduce redundancy.  
- **MetalOxideSensor units** show moderate inter-correlation — they measure different VOCs but from the same air mass.  
- **Temperature and Humidity** have low correlation with most sensors, confirming their limited predictive value as seen in violin plots.  
- **CO_GasSensor** is weakly negatively correlated with CO2 sensors — mechanistically consistent (different combustion byproducts).  
- **Multicollinearity** between CO2 sensors warrants caution for Logistic Regression; tree-based models (Random Forest, XGBoost) are robust to this.


### 5.4 Pairplot — Top Predictive Features

In [ ]:
# Sample for performance
sample = df_clean.sample(1500, random_state=42)
top_features = ["CO2_ElectroChemicalSensor", "MetalOxideSensor_Unit2",
                "MetalOxideSensor_Unit4", "CO_GasSensor", "Activity Level"]

g = sns.pairplot(sample[top_features], hue="Activity Level",
                 palette=PALETTE, plot_kws={"alpha": 0.4, "s": 15},
                 diag_kind="kde",
                 hue_order=["Low Activity", "Moderate Activity", "High Activity"])
g.fig.suptitle("Pairplot — Top 4 Predictive Sensors", y=1.01, fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


**Conclusion — Pairplot**  
- Clear diagonal banding in the `MetalOxideSensor_Unit2` vs `MetalOxideSensor_Unit4` plot confirms these two features together create strong class separation.  
- High Activity cluster is visually distinct in the CO2_ElectroChemical vs MOS space.  
- Low and Moderate Activity overlap substantially — consistent with the class imbalance challenge.  
- This supports using ensemble models capable of learning non-linear boundaries.


---
<a id='6'></a>
## 6. Feature Correlation with Target & Engineered Features

### Purpose
Quantify each feature's relationship to `Activity Level` to understand predictive power, and engineer new features that may improve model performance.


In [ ]:
# ── Encode target numerically for correlation ─────────────────────────────────
activity_enc = {"Low Activity": 0, "Moderate Activity": 1, "High Activity": 2}
df_clean["activity_enc"] = df_clean["Activity Level"].map(activity_enc)

corr_target = df_clean[sensor_cols + ["activity_enc"]].corr()["activity_enc"].drop("activity_enc").sort_values()

fig, ax = plt.subplots(figsize=(9, 5))
colors = ["#E85C5C" if v > 0 else "#4C9BE8" for v in corr_target.values]
bars = ax.barh(corr_target.index, corr_target.values, color=colors, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Pearson Correlation of Sensor Features with Activity Level", fontsize=12)
ax.set_xlabel("Correlation Coefficient")
for bar, val in zip(bars, corr_target.values):
    ax.text(val + (0.005 if val >= 0 else -0.005), bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", ha="left" if val >= 0 else "right", fontsize=9)
plt.tight_layout()
plt.show()


**Conclusion — Feature Correlation with Target**  
| Rank | Feature | Correlation | Interpretation |
|---|---|---|---|
| 1 | MetalOxideSensor_Unit2 | +0.333 | Strong positive — VOC spikes with activity |
| 2 | MetalOxideSensor_Unit4 | +0.311 | Strong positive |
| 3 | CO2_ElectroChemicalSensor | +0.286 | CO2 rises with physical activity |
| 4 | MetalOxideSensor_Unit1 | +0.212 | Moderate positive |
| 5 | MetalOxideSensor_Unit3 | +0.202 | Moderate positive |
| 6 | CO_GasSensor | −0.253 | Negative — CO associated with stationary cooking |
| 7 | CO2_InfraredSensor | −0.128 | Weak negative |
| 8–9 | Temperature, Humidity | ~0 | Negligible direct correlation |

These linear correlations are directional indicators only; tree-based models will capture non-linear interactions automatically.


### 6.1 Feature Engineering

In [ ]:
# ── Engineered features ───────────────────────────────────────────────────────

# 1. CO2 sensor average — two sensors measure same gas; average reduces noise
df_clean["CO2_Average"] = (df_clean["CO2_InfraredSensor"] + df_clean["CO2_ElectroChemicalSensor"]) / 2

# 2. Total MOS activity — combined VOC signal across all 4 MOS units
df_clean["TotalMOS"] = (df_clean["MetalOxideSensor_Unit1"] + df_clean["MetalOxideSensor_Unit2"] +
                        df_clean["MetalOxideSensor_Unit3"] + df_clean["MetalOxideSensor_Unit4"])

# 3. CO2 to CO ratio — captures relative combustion vs respiration balance
df_clean["CO2_CO_Ratio"] = df_clean["CO2_Average"] / (df_clean["CO_GasSensor"] + 1)  # +1 avoids div by zero

# 4. Time of day — ordinal encoding (night < morning < afternoon < evening)
time_order = {"morning": 1, "afternoon": 2, "evening": 3, "night": 0}
df_clean["TimeOfDay_Ordinal"] = df_clean["Time of Day"].map(time_order)

# Verify
print("Engineered features sample:")
df_clean[["CO2_Average", "TotalMOS", "CO2_CO_Ratio", "TimeOfDay_Ordinal", "Activity Level"]].head(8)


In [ ]:
# ── Validate engineered features against target ───────────────────────────────
eng_features = ["CO2_Average", "TotalMOS", "CO2_CO_Ratio"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, eng_features):
    sns.boxplot(data=df_clean, x="Activity Level", y=col,
                order=["Low Activity", "Moderate Activity", "High Activity"],
                palette=PALETTE, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=15)

plt.suptitle("Engineered Features vs Activity Level", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


**Conclusion — Feature Engineering**  
Four features were engineered with the following justifications:

| Feature | Rationale |
|---|---|
| `CO2_Average` | Averages two correlated CO2 sensors to reduce noise while retaining signal; reduces multicollinearity |
| `TotalMOS` | Aggregates all 4 MOS units into a single VOC activity index — the top individual predictors |
| `CO2_CO_Ratio` | Captures the balance between respiratory CO2 (activity) and combustion CO (cooking/sedentary) — a novel discriminative signal |
| `TimeOfDay_Ordinal` | Encodes natural ordering of time periods for models that may benefit from ordinal encoding |

Boxplots confirm `TotalMOS` shows **clear monotonic increase** with activity level — making it a strong engineered predictor.


---
<a id='7'></a>
## 7. Key Findings Summary

### Summary of EDA Insights

**Data Quality Issues Found & Actions:**
| Issue | Detail | Action |
|---|---|---|
| Dirty Activity Level labels | 6 variants → 3 classes | Normalise via mapping |
| Dirty HVAC labels | 23 variants → 6 classes | Lowercase + strip normalisation |
| Impossible Temperature values | 795 readings > 100°C | Cap at 40°C, replace with median |
| Impossible Humidity values | 207 negative, 207 > 100% | Cap to [0, 100], replace with median |
| Missing values | Humidity 19.3%, MOS_Unit2 14.1%, Ambient 10.5%, CO 8.3% | Median/mode imputation |
| Class imbalance | High Activity only 10.9% | SMOTE + class_weight='balanced' |

**Key Predictors (ranked):**
1. `MetalOxideSensor_Unit2` & `Unit4` — strongest positive correlation with activity
2. `CO2_ElectroChemicalSensor` — reliable activity proxy via respiration
3. `CO_GasSensor` — useful inverse signal (sedentary cooking pattern)
4. `TotalMOS` (engineered) — strongest single discriminative feature
5. `Time of Day`, `Ambient Light Level` — meaningful categorical context

**Features with low predictive value:**
- Raw `Temperature` and `Humidity` — retain but expect low feature importance
- `Session ID` — identifier only; will be dropped from modelling

**Model Selection Implications:**
- Non-linear boundaries between classes → tree-based models preferred
- Multicollinearity between CO2 sensors → regularisation needed for Logistic Regression
- Class imbalance → avoid accuracy as sole metric; use **weighted F1-score**
- Feature scale variance → standardise for Logistic Regression

**Recommended Models for Pipeline:**
1. **Random Forest** — robust baseline, feature importance, handles imbalance
2. **XGBoost** — strong tabular learner, scale-insensitive, tunable
3. **Logistic Regression** — interpretable linear baseline for comparison


In [ ]:
print("EDA Complete.")
print(f"Clean dataset shape: {df_clean.shape}")
print(f"Target classes: {df_clean['Activity Level'].unique()}")
print(f"Features available for modelling: {df_clean.shape[1] - 2} (excl. raw target and encoded target)")
